# Facial Recognition in Historical Photographs with AI in Python

**Source lesson:** [Programming Historian — Facial Recognition in Historical Photographs with Artificial Intelligence in Python](https://programminghistorian.org/en/lessons/facial-recognition-ai-python) by Charles Goldberg and Zach Haala (CC-BY 4.0).

This notebook is my working version of that lesson: it walks through the same steps, but the code cells are **fill-in-the-blank** — I write the code myself, using the lesson's explanations as a guide.

**Run this in Google Colab.** It needs `dlib`, `TensorFlow`, a C++ compiler, and works best with GPU access — all of which Colab provides out of the box. Running locally would require setting all of that up manually.

**Dataset:** The lesson's own dataset (Bethel University yearbooks via a Google Drive zip, or via Bethel's digital library) is dead as of November 2026 — see the note in Step 1 below for details. I substituted **Simmons University's "Microcosm" yearbook**, pulled from the Internet Archive, using the exact same years (1911, 1921, 1931, 1941, 1951, 1961) so the lesson's year-over-year structure still applies:
- [1911](https://archive.org/download/microcosm1911simm/microcosm1911simm.pdf) · [1921](https://archive.org/download/microcosm1921simm/microcosm1921simm.pdf) · [1931](https://archive.org/download/microcosm1931simm/microcosm1931simm.pdf) · [1941](https://archive.org/download/microcosm1941simm/microcosm1941simm.pdf) · [1951](https://archive.org/download/microcosm1951simm/microcosm1951simm.pdf) · [1961](https://archive.org/download/microcosm1961simm/microcosm1961simm.pdf)
- The OpenCV Haar Cascade face-detection model (`haarcascade_frontalface_default.xml`) ships inside the `opencv-python` package itself (accessed via `cv2.data.haarcascades`), so it doesn't need a separate download.

**Note:** Simmons was an all-women's undergraduate college throughout this entire period (founded 1899 as "Simmons Female College"), unlike Bethel, which was majority male. That's a meaningful difference, not just a cosmetic dataset swap — see the updated ethics note below.


In [ ]:
from google.colab import _message
import json

nb = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
with open('Wolf-FacialRecognition-Notebook.ipynb', 'w') as f:
    json.dump(nb, f)

!jupyter nbconvert --to html Wolf-FacialRecognition-Notebook.ipynb

from google.colab import files
files.download('Wolf-FacialRecognition-Notebook.ipynb')
files.download('Wolf-FacialRecognition-Notebook.html')

## Learning Outcomes

By the end of this notebook I should be able to:
- Explain the basics of computer vision machine learning
- Explain how a computer "views" an image (pixels → RGB values)
- Extract pictures of human faces from historical documents digitized as PDFs, and organize them for further analysis
- Apply a pre-trained computer vision model (Haar Cascade, then DeepFace) to a historical dataset

## Prior Skills Assumed
- Basic Python 3 familiarity
- A Google account (for Colab)
- Basic familiarity with Jupyter/Colab notebooks


## A Note on Ethics Before I Start

This lesson explicitly warns that facial recognition and emotion-detection ML models are **not neutral tools**. Before running any code, it's worth sitting with why:

- Object recognition models can fail to detect or misidentify non-white and female faces more often than white male faces, partly because of how training data has historically been assembled.
- This specific dataset (Simmons University's "Microcosm" yearbook) is majority white and entirely female at the undergraduate level throughout this period — Simmons was a women's-only college from its 1899 founding through at least 1961. That's a different skew than the lesson's original Bethel dataset (majority white and male), and it limits how representative any findings will be in a different direction.
- Historical photos differ from the modern photos most models are trained on (poor quality, faded, rescanned), which can compound bias.

**Reflection question 1:** *Before running any code, what's one reason a facial-recognition model trained mostly on modern, high-quality photos of white faces might struggle with a 1911 yearbook scan? Write 2-3 sentences.*

> Your answer here: Probably both the overall picture quality also without color does not offer as many relevant tokens for the model to recognize and also the model is trained to recognize with those extra more relevant tokens.
>
>


## Step 1: Lesson Setup

This first cell needs to:
1. Create a working folder called `yearbook` and move into it
2. Install `gdown` (a tool for downloading files from Google Drive) and use it to download the lesson's zipped dataset
3. Unzip the dataset and create an `images` subfolder
4. Install the ML/PDF libraries this lesson depends on: `PyMuPDF` (PDF → image conversion), `dlib` and `DeepFace` (face/emotion detection); OpenCV (`cv2`) comes bundled with `DeepFace`'s dependencies
5. Import everything needed and disable GPU use (Colab + TensorFlow compatibility workaround)

**TODO:** Fill in the blanks marked `# TODO`. The Google Drive file ID for the dataset is `1NHT8NN8ClBEnUC5VqkP3wr2KhyiIQzyU`.

**Update (dead dataset):** Both of the lesson's original data sources are gone as of November 2026:
- The zipped Google Drive dataset (`PHfiles.zip`, file ID above) returns a 404 — the file no longer exists on Drive.
- Bethel University's own Digital Library, which the live lesson page also links to as a manual-download fallback, now returns "The collection 'p16120coll2' cannot be displayed. Log in and refresh the page to access restricted or unpublished collections." — the collection itself has since become access-restricted.

This is a common hazard with tutorials that depend on third-party file hosting outside of their own site. After checking the lesson's GitHub repo too (it only hosts the notebook and the Haar Cascade file, no PDFs), I substituted **Simmons University's "Microcosm" yearbook**, freely downloadable from the Internet Archive, using the exact same years so the rest of the lesson's structure (and the year-over-year table in Step 6) still applies as a point of comparison:

- [1911](https://archive.org/download/microcosm1911simm/microcosm1911simm.pdf)
- [1921](https://archive.org/download/microcosm1921simm/microcosm1921simm.pdf)
- [1931](https://archive.org/download/microcosm1931simm/microcosm1931simm.pdf)
- [1941](https://archive.org/download/microcosm1941simm/microcosm1941simm.pdf)
- [1951](https://archive.org/download/microcosm1951simm/microcosm1951simm.pdf)
- [1961](https://archive.org/download/microcosm1961simm/microcosm1961simm.pdf)

I downloaded each, renamed it to `<year>.pdf` (the code below splits filenames on `.` to get the year, so the name has to match exactly), and uploaded them directly into the `yearbook/` folder in Colab rather than downloading+unzipping a bundle.

In [1]:
%mkdir yearbook
%cd yearbook

# NOTE: the lesson's gdown/zip download is dead (404) as of November 2026 — see the
# markdown note above. Downloading the six renamed Simmons "Microcosm" yearbook PDFs
# directly from the Internet Archive instead, one per year.

!wget -q -O 1911.pdf https://archive.org/download/microcosm1911simm/microcosm1911simm.pdf
!wget -q -O 1921.pdf https://archive.org/download/microcosm1921simm/microcosm1921simm.pdf
!wget -q -O 1931.pdf https://archive.org/download/microcosm1931simm/microcosm1931simm.pdf
!wget -q -O 1941.pdf https://archive.org/download/microcosm1941simm/microcosm1941simm.pdf
!wget -q -O 1951.pdf https://archive.org/download/microcosm1951simm/microcosm1951simm.pdf
!wget -q -O 1961.pdf https://archive.org/download/microcosm1961simm/microcosm1961simm.pdf
!ls -la *.pdf

%mkdir images

!pip install PyMuPDF
!pip install dlib
!pip install DeepFace

import os, shutil, fitz, cv2, numpy as np, pandas as pd, dlib, tensorflow as tf
from os.path import dirname, join
from deepface import DeepFace

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


/content/yearbook
-rw-r--r-- 1 root root  9555663 Nov 26  2023 1911.pdf
-rw-r--r-- 1 root root 19121793 Nov 26  2023 1921.pdf
-rw-r--r-- 1 root root 17185905 Nov 26  2023 1931.pdf
-rw-r--r-- 1 root root 11260705 Nov 26  2023 1941.pdf
-rw-r--r-- 1 root root  9162229 Nov 26  2023 1951.pdf
-rw-r--r-- 1 root root 15570479 Nov 26  2023 1961.pdf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 4.3 MB/s  0:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 10.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [PyMuPDF]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 MB 30.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [DeepFace]


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


26-11-02 22:26:22 - Directory /root/.deepface has been created
26-11-02 22:26:22 - Directory /root/.deepface/weights has been created


## Step 2: PDF Conversion

Now I'll convert each yearbook PDF into individual page images. For each PDF file (e.g. `1911.pdf`) this step should:

1. Find every file ending in `.pdf` in the current folder
2. Create a subfolder inside `/images` named after the PDF (e.g. `/images/1911`)
3. Copy the PDF into its new subfolder
4. Use PyMuPDF (`fitz`) to open the PDF and save each page as a separate `.png` image (e.g. `page-0.png`, `page-1.png`, ...)

**Hints:**
- `[f for f in os.listdir(path) if f.endswith('.pdf')]` gives you the list of PDF filenames
- `pdf.split(".")[0]` strips `.pdf` off a filename, leaving just the year
- `shutil.copy(src, dst)` copies a file
- `fitz.open(pdf)` opens a PDF; looping `for page in doc:` gives you each page
- `page.get_pixmap()` renders a page to an image object; `.save("page-%i.png" % page.number)` writes it to disk
- Use `os.chdir(...)` to move between folders as you go, and `os.chdir("..")` / `os.chdir(os.path.dirname(os.getcwd()))` to move back up

In [2]:
path = r'./'

pdfs = [f for f in os.listdir(path) if f.endswith('.pdf')]

for pdf in pdfs:
    os.chdir(os.path.join('./images'))
    os.mkdir((pdf.split(".")[0]))
    newdir = (os.path.join('./images/' + os.path.join(pdf.split(".")[0])))
    os.chdir("..")

    print("Now copying images into " + (newdir))
    shutil.copy(pdf, newdir)

    os.chdir(newdir)
    doc = fitz.open(pdf)

    for page in doc:
        pix = page.get_pixmap()
        pix.save("page-%i.png" % page.number)

    os.chdir(os.path.dirname(os.getcwd()))
    os.chdir("..")


Now copying images into ./images/1911
Now copying images into ./images/1921
Now copying images into ./images/1941
Now copying images into ./images/1931
Now copying images into ./images/1961
Now copying images into ./images/1951


## Step 3: Fundamentals of Object Detection (read + reflect, no code)

The lesson explains that:
- A computer sees an image as a grid of pixels, each represented by three numbers (R, G, B) from 0-255
- To save processing power, images are often **downscaled** and converted to **greyscale** before analysis ("thresholding")
- A **Haar Cascade** is a fast, older (2001) object-detection technique that looks for patterns of pixel groups rather than every individual pixel, using many "weak classifiers" that combine into a "strong classifier" when enough of them agree a face is present

**Reflection question 2:** *Why would greyscale conversion and downscaling make a Haar Cascade faster, at the cost of what kind of information?*

> Your answer here: Because it would limit the amount of pixels you would have to use, and it would cost us slight details.
>
>

**Reflection question 3:** *The lesson notes that lighter skin tones tend to produce greater pixel-density differences than darker skin tones for a computer to detect. Given that this dataset (Simmons University's "Microcosm" yearbook) is majority white throughout, what effect might that have on the face-detection counts you get in Step 4, independent of any bias in the algorithm itself?*

> Your answer here: The dataset after the algorithm will represent a heavily white population, and other races will serve mostly as small footnotes.
>
>


## Step 4: Processing the Images (Face Detection)

Now I'll walk through every `.png` page image and use OpenCV's Haar Cascade to find and crop out individual faces, saving each one into a new `/<year> faces` folder.

For each year's folder, this step should:
1. List every `.png` file in that folder
2. If a `/<year> faces` subfolder doesn't already exist, create it
3. For each `.png` page image:
   - Read it in with OpenCV
   - Convert it to greyscale
   - Load OpenCV's pretrained frontal-face Haar Cascade (`cv2.data.haarcascades + "haarcascade_frontalface_default.xml"`)
   - Run `detectMultiScale` on the greyscale image to get bounding boxes `(x, y, w, h)` for each detected face
   - Crop each face out (with a little padding so the crop isn't too tight), draw a rectangle on it, and save it as a new `.png` in the `/<year> faces` folder
   - Delete the original full-page `.png` once its faces have been extracted

**Hints:**
- `cv2.imread(png)` reads an image; `cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)` converts it to greyscale
- `cv2.CascadeClassifier(...)` loads the Haar Cascade model
- `face_cascade.detectMultiScale(image=..., scaleFactor=1.9, minNeighbors=4)` returns a list of `(x, y, w, h)` boxes
- Slicing `image[y1:y2, x1:x2]` crops a NumPy image array
- Wrap the crop in a `try/except` — not every detected box will crop cleanly

In [3]:
path = r'./'

os.chdir(os.path.join(path + 'images'))
dirs = os.listdir(path)
for dir in dirs:
    os.chdir(os.path.join(path + dir))
    pngs = [f for f in os.listdir(path) if f.endswith('.png')]

    if not os.path.exists((dir) + ' faces'):
        print("New 'faces' directory created in " + (dir) + " folder")
        os.makedirs((dir) + ' faces')

        count = 0
        for png in pngs:
            image = cv2.imread(png)

            greyscale_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

            detected_faces = face_cascade.detectMultiScale(image=greyscale_image, scaleFactor=1.9, minNeighbors=4)

            count = 0
            for (x, y, w, h) in detected_faces:
                try:
                    xpadding = 20
                    ypadding = 40
                    crop_face = image[y-ypadding: y + h+ypadding, x-xpadding: x + w+xpadding]
                    count += 1
                    face = cv2.rectangle(crop_face, (x, y), (x+w, y+h), (255, 0, 0), 2)
                    cv2.imwrite(path + (dir) + ' faces/' + str(count) + '_' + png, face)
                except (Exception):
                    print("An error happened")
                    continue
            os.remove(os.path.join(path, png))
    os.chdir("..")


New 'faces' directory created in 1941 folder
An error happened
An error happened
An error happened
An error happened
An error happened
New 'faces' directory created in 1951 folder
An error happened
New 'faces' directory created in 1931 folder
An error happened
New 'faces' directory created in 1911 folder
New 'faces' directory created in 1961 folder
New 'faces' directory created in 1921 folder


## Step 5: Identifying Smiles

Haar Cascades are fast but rigid — they only recognize what they were explicitly trained to recognize. For emotion detection, this lesson switches to **DeepFace**, which uses a Convolutional Neural Network (CNN): a "deep learning" model made of layered "neurons" that each learn to recognize small patterns (an eye corner, an ear lobe, ...) and pass their findings to the next layer.

A photo is treated as containing a smile if DeepFace's dominant predicted emotion is `'happy'`; anything else counts as a non-smile.

This step should, for each year:
1. Walk through the `<year> faces` folder and collect every cropped face `.png`
2. Run `DeepFace.analyze(png, actions=['emotion'])` on each one
3. Tally how many are `'happy'` (smiles) vs. anything else (non-smiles), and how many raise an error
4. Convert each year's tallies into ratios (smiles / total, non-smiles / total, errors / total)
5. Save the results to a `YearbookOutput.csv` file with columns: `Years`, `Smiles`, `Non-Smiles`, `Error Weight`

**Reflection question 4:** *DeepFace's emotion classifier reduces every face to one of a handful of labeled emotions, with 'happy' as our stand-in for 'smiling.' What's a case where that mapping might misclassify a real historical photograph?*

> Your answer here: Emotions are always subjective and can be misinterpreted — a smile could really be a nervous grimace, and fake smiles and fake frowns will exist in lots of contexts as well.
>
>


In [4]:
%cd ..
number_smiles = 0
smile_counts = []
number_nonsmiles = 0
nonsmile_counts = []
num_errors = 0
error_counts = []
pngs = []

file_count = 0
file_count_list = []
iterator = 0
count = 0
total_loops = 0

years = ['1911', '1921', '1931', '1941', '1951', '1961']

for year in years:
    path = r'./images' + '/' + year
    for root, dirs, files in os.walk(path):
        for _ in dirs:
            path = path + '/' + (year + ' faces')
            if (file_count != 0):
                file_count_list.append(file_count)
            file_count = 0
            for f in os.listdir(path):
                if f.endswith('.png'):
                    pngs.append(path + '/' + f)
                    file_count = file_count + 1

for png in pngs:
    try:
        total_loops = total_loops + 1
        count = count + 1
        if (count != (file_count_list[iterator] + 1)):
            demography = DeepFace.analyze(png, actions=['emotion'])
            print(demography)
            if (demography[0]['dominant_emotion'] == 'happy'):
                number_smiles = number_smiles + 1
            else:
                number_nonsmiles = number_nonsmiles + 1
        else:
            count = count - 1
            smile_counts.append(number_smiles / count)
            nonsmile_counts.append(number_nonsmiles / count)
            error_counts.append(num_errors / count)
            number_smiles = 0
            number_nonsmiles = 0
            num_errors = 0
            iterator = iterator + 1
            count = 0
    except (Exception):
        num_errors = num_errors + 1
        continue

smile_counts.append(number_smiles / count)
nonsmile_counts.append(number_nonsmiles / count)
error_counts.append(num_errors / count)

header = {'Years': years, 'Smiles': smile_counts, 'Non-Smiles': nonsmile_counts, "Error Weight": error_counts}
data = pd.DataFrame(header)
data.to_csv('YearbookOutput.csv', index=False)


/content/yearbook
26-11-02 22:40:33 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 76.4MB/s]


[{'emotion': {'angry': np.float32(0.69983625), 'disgust': np.float32(4.8108664e-09), 'fear': np.float32(0.0014975576), 'happy': np.float32(92.42669), 'sad': np.float32(0.14961916), 'surprise': np.float32(0.0031207523), 'neutral': np.float32(6.71924)}, 'dominant_emotion': 'happy', 'region': {'x': 26, 'y': 47, 'w': 72, 'h': 72, 'left_eye': None, 'right_eye': None}, 'face_confidence': 0.92}]
[{'emotion': {'angry': np.float32(2.5095975), 'disgust': np.float32(0.0006277259), 'fear': np.float32(0.7946011), 'happy': np.float32(0.018324228), 'sad': np.float32(4.487638), 'surprise': np.float32(0.010536253), 'neutral': np.float32(92.17868)}, 'dominant_emotion': 'neutral', 'region': {'x': 29, 'y': 46, 'w': 70, 'h': 70, 'left_eye': None, 'right_eye': None}, 'face_confidence': 0.91}]
[{'emotion': {'angry': np.float32(0.082726255), 'disgust': np.float32(4.9743385e-06), 'fear': np.float32(0.025846187), 'happy': np.float32(0.022464337), 'sad': np.float32(2.498891), 'surprise': np.float32(0.0010476604)

## Step 6: Download and Results

In Colab, download the finished CSV so I can open it in a spreadsheet app.

In [5]:
from google.colab import files
files.download('YearbookOutput.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Interpreting My Results

The lesson's own run — on the original Bethel dataset, which is majority male — found smiling-face rates roughly like this. My run is on Simmons' all-women's "Microcosm" yearbook instead (see the Step 1 note), so differences between these numbers and mine could come from the different institution, the different gender composition, or just normal run-to-run noise in DeepFace — not a direct apples-to-apples comparison:

| Year | Smiles | Non-Smiles | Error Weight |
|------|--------|-----------|---------------|
| 1911 | 0%     | 100%      | 0%            |
| 1921 | 0%     | 92%       | 8%            |
| 1931 | 15%    | 70%       | 30% (interp.)  |
| 1941 | 3%     | 40%       | 53%           |
| 1951 | 25%    | 68%       | 14%           |
| 1961 | 43%    | 53%       | 9%            |

**Reflection question 5:** *How do your own `YearbookOutput.csv` numbers compare to the table above? If 1941 is an outlier in your run too, what did the lesson suggest as the likely explanation (hint: look at what kind of photos dominate that yearbook)?*

> Your answer here:
>
>

**Reflection question 6:** *The lesson connects the rising smile rate to broader social changes (e.g. the shift toward coeducation accelerating in the 1960s-70s). What's one alternative, non-social explanation for a rising smile rate over time that has nothing to do with who the students were (hint: think about camera/photography technology itself)?*

> Your answer here:
>
>


## Conclusions and Next Steps

Some directions to explore further, per the lesson:
- Training a custom object-recognition model instead of relying on pre-trained ones (see the Programming Historian lesson *Computer Vision for the Humanities: An Introduction to Deep Learning for Image Classification*)
- Applying the same "distant viewing" approach to a larger, more demographically diverse dataset
- Exploring other pre-trained models via Hugging Face, or point-and-click trainers like Roboflow

**Reflection question 7 (wrap-up):** *In 2-4 sentences, what did this exercise teach you about the tradeoff between how easy pre-trained ML models are to apply to historical images, and how trustworthy their output is as historical evidence?*

> Your answer here: I think my main reflection is that this technology can be extremely useful for making datasets fast, however the datasets they make won't be perfect. If you were to use their output as historical evidence, you would have to acknowledge the certain unreliabilities that the dataset would have in your argument.
>
>
